In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

####Orders

In [0]:
df_bronze = spark.read.table('ecom.bronze.orders')

df_silver = df_bronze.withColumn('order_status', lower('order_status')) #standardizing order status
df_silver = df_silver.withColumn('order_purchase_timestamp', to_timestamp(trim('order_purchase_timestamp')))\
    .withColumn('order_approved_at', to_timestamp('order_approved_at'))\
    .withColumn('order_delivered_carrier_date', to_timestamp('order_delivered_carrier_date'))\
    .withColumn('order_delivered_customer_date', to_timestamp('order_delivered_customer_date'))\
    .withColumn('order_estimated_delivery_date', to_timestamp('order_estimated_delivery_date'))\
    .withColumn('_silver_time', current_timestamp())

#######################################################################################################

df_silver = df_silver.dropna(subset=["order_id", "customer_id", "order_purchase_timestamp"])

df_silver = df_silver.dropDuplicates(["order_id"])

df_silver = df_silver.drop('_ingestion_time', ' _source_file')

df_silver = df_silver.withColumn(
    "_quality_flag",
    when(col("order_approved_at").isNull(), "missing_approval")
    .when(col("order_delivered_customer_date").isNull(), "not_delivered")
    .otherwise("ok")
)

In [0]:
if spark.catalog.tableExists('ecom.silver.orders'):
    trg_obj = DeltaTable.forName(spark,'ecom.silver.orders')
    trg_obj.alias('trg').merge(df_silver.alias('src'),'src.order_id = trg.order_id')\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
else:
    df_silver.write.format('delta').saveAsTable('ecom.silver.orders')


In [0]:
display(spark.read.table("ecom.silver.orders").limit(10))

####Order_items

In [0]:
df_bronze = spark.read.table('ecom.bronze.order_items')

df_silver = df_bronze.withColumn("price", col("price").cast("double")) \
                      .withColumn("freight_value", col("freight_value").cast("double")) \
                      .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))

df_silver = df_silver.drop('_ingestion_time','_source_file')
df_silver = df_silver.dropDuplicates(['order_item_id'])

In [0]:
if spark.catalog.tableExists('ecom.silver.orders_items'):
    trg_obj = DeltaTable.forName(spark,'ecom.silver.orders_items')
    trg_obj.alias('trg').merge(df_silver.alias('src') ,'src.order_item_id= trg.order_item_id')\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
else:
    df_silver.write.format('delta').saveAsTable('ecom.silver.orders_items')

####order_payments

In [0]:
df_bronze = spark.read.table('ecom.bronze.order_payments')

df_silver = df_bronze.withColumn("payment_value", col("payment_value").cast("double")) \
                      .withColumn("payment_installments", col("payment_installments").cast("integer"))

df_silver = df_silver.drop('_ingestion_time','_source_file')

df_silver = df_silver.dropDuplicates(['payment_sequential','order_id'])

In [0]:
if spark.catalog.tableExists('ecom.silver.orders_payments'):
    trg_obj = DeltaTable.forName(spark,'ecom.silver.orders_payments')
    trg_obj.alias('trg').merge(df_silver.alias('src') ,'src.order_id= trg.order_id AND src.payment_sequential= trg.payment_sequential')\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
else:
    df_silver.write.format('delta').saveAsTable('ecom.silver.orders_payments')

####customers

In [0]:
df_bronze = spark.read.table('ecom.bronze.customers')

df_silver = df_bronze.drop('_ingestion_time','_source_file')
df_silver = df_silver.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("integer"))

df_silver = df_silver.dropDuplicates(['customer_id','customer_unique_id'])

In [0]:
if spark.catalog.tableExists('ecom.silver.customers'):
    trg_obj = DeltaTable.forName(spark,'ecom.silver.customers')
    trg_obj.alias('trg').merge(df_silver.alias('src'),'src.customer_unique_id= trg.customer_unique_id AND src.customer_id= trg.customer_id')\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
else:
    df_silver.write.format('delta').saveAsTable('ecom.silver.customers')

In [0]:
# tables = ["ecom.silver.customers", "ecom.silver.orders",
#           "ecom.silver.orders_items", "ecom.silver.orders_payments"]

# for table in tables:
#     count = spark.read.table(table).count()
#     print(f"{table}: {count} records")

####products

In [0]:
df_bronze = spark.read.table('ecom.bronze.products')

df_silver = df_bronze.drop('_ingestion_time','_source_file')
df_silver = df_silver.dropDuplicates(['product_id'])

df_silver = df_silver.withColumn("product_name_lenght", col("product_name_lenght").cast("double"))\
                      .withColumn("product_description_lenght", col("product_description_lenght").cast("double"))\
                      .withColumn("product_photos_qty", col("product_photos_qty").cast("integer"))\
                      .withColumn("product_weight_g", col("product_weight_g").cast("double"))\
                      .withColumn("product_length_cm", col("product_length_cm").cast("double"))\
                      .withColumn("product_height_cm", col("product_height_cm").cast("double"))\
                      .withColumn("product_width_cm", col("product_width_cm").cast("double"))

In [0]:
if spark.catalog.tableExists('ecom.silver.products'):
    trg_obj = DeltaTable.forName(spark,'ecom.silver.products')
    trg_obj.alias('trg').merge(df_silver.alias('src'),'src.product_id= trg.product_id')\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
else:
    df_silver.write.format('delta').saveAsTable('ecom.silver.products')